# Vesuvius V8 - Multi-Scale Hierarchical Training

## Design Philosophy
- **Early layers (128³)**: Fine papyrus detail → SurfaceDice
- **Mid layers (pool to 64³→32³)**: Global sheet structure → VOI
- **Deep reasoning**: Topology preservation → TopoScore

## Target: 600 epochs, break 0.550 LB

## Key Optimizations
1. **128³ patches** - faster than 192³, better fine detail
2. **Multi-scale encoder** - hierarchical feature extraction
3. **Early topology losses** - skeleton from epoch 0
4. **OneCycleLR** - faster convergence than polynomial decay

In [ ]:
!pip install imagecodecs connected-components-3d -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Data
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints")
    
    # ==========================================================================
    # MULTI-SCALE ARCHITECTURE
    # ==========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)  # Faster, fine detail
    FEATURES: List[int] = None      # [32, 64, 128, 256, 320] - 5 stages
    N_BLOCKS: List[int] = None      # [1, 2, 3, 4, 4]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # ==========================================================================
    # FAST TRAINING - 600 EPOCHS
    # ==========================================================================
    EPOCHS: int = 600
    BATCH_SIZE: int = 6             # Larger batch with 128³
    NUM_WORKERS: int = 16
    PREFETCH_FACTOR: int = 4
    
    # OneCycleLR for fast convergence
    MAX_LR: float = 0.01
    DIV_FACTOR: float = 25.0        # start_lr = max_lr/25
    FINAL_DIV_FACTOR: float = 1000.0
    PCT_START: float = 0.1          # 10% warmup
    
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # ==========================================================================
    # TOPOLOGY-FOCUSED LOSSES (from epoch 0)
    # ==========================================================================
    DICE_WEIGHT: float = 0.2
    BCE_WEIGHT: float = 0.1
    SKELETON_WEIGHT: float = 0.35
    CLDICE_WEIGHT: float = 0.35
    
    # All topology losses active from start!
    SKELETON_START_EPOCH: int = 0
    SKELETON_WARMUP_EPOCHS: int = 100
    CLDICE_START_EPOCH: int = 50
    CLDICE_WARMUP_EPOCHS: int = 100
    
    # Training control
    RESUME_TRAINING: bool = True
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 20
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # Data
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 12    # More patches per volume with 128³
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320]  # 5 stages (faster)
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 2, 3, 4, 4]          # Fewer blocks
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()
cfg.__post_init__()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

print("="*60)
print("V8 MULTI-SCALE FAST TRAINING")
print("="*60)
print(f"Patch: {cfg.PATCH_SIZE} | Batch: {cfg.BATCH_SIZE}")
print(f"Epochs: {cfg.EPOCHS} | OneCycleLR max={cfg.MAX_LR}")
print(f"Stages: {len(cfg.FEATURES)} | Features: {cfg.FEATURES}")
print(f"Topology from epoch 0: skel@{cfg.SKELETON_START_EPOCH}, clDice@{cfg.CLDICE_START_EPOCH}")
print("="*60)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# CELL 2: MULTI-SCALE MODEL
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    """Safe InstanceNorm for AMP."""
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, x):
        mean = x.mean(dim=[2,3,4], keepdim=True)
        var = x.var(dim=[2,3,4], keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var.clamp(min=self.eps) + self.eps)
        if self.affine:
            x = self.weight.view(1,-1,1,1,1) * x + self.bias.view(1,-1,1,1,1)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size, padding=kernel_size//2, bias=False),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    
    def forward(self, x):
        return x + self.blocks(x)


class MultiScaleBlock(nn.Module):
    """
    Multi-scale feature extraction block.
    Captures both fine (3x3) and coarse (5x5) features.
    """
    def __init__(self, channels):
        super().__init__()
        half = channels // 2
        
        # Fine branch (3x3)
        self.fine = nn.Sequential(
            nn.Conv3d(channels, half, 3, padding=1, bias=False),
            SafeInstanceNorm3d(half),
            nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Coarse branch (5x5 via dilated 3x3)
        self.coarse = nn.Sequential(
            nn.Conv3d(channels, half, 3, padding=2, dilation=2, bias=False),
            SafeInstanceNorm3d(half),
            nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Fusion
        self.fuse = nn.Sequential(
            nn.Conv3d(channels, channels, 1, bias=False),
            SafeInstanceNorm3d(channels),
        )
    
    def forward(self, x):
        fine = self.fine(x)
        coarse = self.coarse(x)
        merged = torch.cat([fine, coarse], dim=1)
        return x + self.fuse(merged)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b,c))))).view(b,c,1,1,1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class MultiScaleUNet3D(nn.Module):
    """
    Multi-Scale Hierarchical UNet3D.
    
    - Early layers: Fine 128³ detail (3x3 convs)
    - Mid layers: Multi-scale blocks (3x3 + dilated)
    - Deep layers: Global context (pooled to 8³)
    """
    
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_scse=True, use_deep_supervision=True):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320]
        if n_blocks is None:
            n_blocks = [1, 2, 3, 4, 4]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            
            # Build encoder stage
            layers = [ConvBlock(in_channels, feat)]
            for j in range(nb):
                if i >= 2:  # Use multi-scale in deeper layers
                    layers.append(MultiScaleBlock(feat))
                else:
                    layers.append(ResBlock(feat))
            
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            self.dec_convs.append(ConvBlock(features[i]*2, features[i]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) 
                for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Test
_m = MultiScaleUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS)
print(f"Model: {count_params(_m)/1e6:.1f}M params")
del _m

In [ ]:
# =============================================================================
# CELL 3: LOSS FUNCTIONS
# =============================================================================

def soft_skeletonize(x, num_iter=10):
    """Soft skeletonization via iterative min-max pooling."""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class CombinedLoss(nn.Module):
    def __init__(self, dice_w=0.2, bce_w=0.1, skel_w=0.35, cldice_w=0.35,
                 skel_start=0, skel_warmup=100, cldice_start=50, cldice_warmup=100):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w = bce_w
        self.skel_w = skel_w
        self.cldice_w = cldice_w
        
        self.skel_start = skel_start
        self.skel_warmup = skel_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        self.ds_weights = [0.5, 0.25, 0.125]
    
    def _scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def dice_loss(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        inter = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * inter + 1e-5) / (union + 1e-5)
    
    def bce_loss(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            return (loss * valid).sum() / (valid.sum() + 1e-8)
        return F.binary_cross_entropy_with_logits(pred, target)
    
    def skeleton_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=10)
        target_tube = F.max_pool3d(target_skel, 5, stride=1, padding=2)
        recall = (pred_sig * target_tube).sum() / (target_tube.sum() + 1e-5)
        return 1 - recall
    
    def cldice_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            pred_sig = pred_sig * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred_sig, 10)
        skel_target = soft_skeletonize(target, 10)
        
        tprec = ((skel_pred * target).sum() + 1e-5) / (skel_pred.sum() + 1e-5)
        tsens = ((skel_target * pred_sig).sum() + 1e-5) / (skel_target.sum() + 1e-5)
        return 1 - 2 * tprec * tsens / (tprec + tsens + 1e-5)
    
    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep = output.get('deep', [])
        else:
            pred = output
            deep = []
        
        skel_scale = self._scale(epoch, self.skel_start, self.skel_warmup)
        cldice_scale = self._scale(epoch, self.cldice_start, self.cldice_warmup)
        
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        skel = self.skeleton_loss(pred, target, ignore_mask) if skel_scale > 0 else torch.tensor(0.0, device=pred.device)
        cldice = self.cldice_loss(pred, target, ignore_mask) if cldice_scale > 0 else torch.tensor(0.0, device=pred.device)
        
        total = (self.dice_w * dice + self.bce_w * bce + 
                 self.skel_w * skel_scale * skel + 
                 self.cldice_w * cldice_scale * cldice)
        
        # Deep supervision
        for i, ds in enumerate(deep):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds.shape[2:], mode='nearest')
            total = total + self.ds_weights[i] * self.dice_loss(ds, ds_target, ds_mask)
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skel': skel.item() if skel_scale > 0 else 0,
            'cldice': cldice.item() if cldice_scale > 0 else 0,
        }

print("Loss functions ready")
print(f"  Skeleton: epoch {cfg.SKELETON_START_EPOCH} + {cfg.SKELETON_WARMUP_EPOCHS} warmup")
print(f"  clDice: epoch {cfg.CLDICE_START_EPOCH} + {cfg.CLDICE_WARMUP_EPOCHS} warmup")

In [ ]:
# =============================================================================
# CELL 4: DATASET
# =============================================================================

def load_volume(path):
    try:
        import tifffile
        return tifffile.imread(str(path))
    except:
        im = Image.open(path)
        return np.stack([np.array(p) for p in ImageSequence.Iterator(im)], axis=0)


class VesuviusDataset(Dataset):
    def __init__(self, csv_path, images_dir, labels_dir, patch_size=(128,128,128),
                 is_train=True, data_fraction=1.0, patches_per_volume=12, preload=True):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        df = pd.read_csv(csv_path)
        valid_ids = [i for i in df['id'].values 
                     if (self.images_dir/f"{i}.tif").exists() and (self.labels_dir/f"{i}.tif").exists()]
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.volumes = {}
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(valid_ids)} volumes...")
            for vid in tqdm(valid_ids, desc="Loading"):
                img = load_volume(self.images_dir/f"{vid}.tif").astype(np.float32)
                lbl = load_volume(self.labels_dir/f"{vid}.tif").astype(np.uint8)
                img = (img - img.mean()) / (img.std() + 1e-8)
                self.volumes[vid] = (img, lbl)
                
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vid] = fg if len(fg) > 0 else None
            
            gb = sum(v[0].nbytes + v[1].nbytes for v in self.volumes.values()) / 1e9
            print(f"Loaded {len(self.volumes)} volumes ({gb:.1f} GB)")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vid = self.volume_ids[idx // self.patches_per_volume]
        img, lbl = self.volumes[vid]
        
        d, h, w = img.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            img = np.pad(img, ((0,max(0,pd-d)),(0,max(0,ph-h)),(0,max(0,pw-w))), mode='reflect')
            lbl = np.pad(lbl, ((0,max(0,pd-d)),(0,max(0,ph-h)),(0,max(0,pw-w))), mode='constant', constant_values=2)
            d, h, w = img.shape
        
        # Sample position (60% foreground-centered)
        fg = self.fg_coords.get(vid)
        if self.is_train and random.random() < 0.6 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_p = img[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_p = lbl[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Augmentation
        if self.is_train:
            for ax in range(3):
                if random.random() > 0.5:
                    img_p = np.flip(img_p, ax)
                    lbl_p = np.flip(lbl_p, ax)
            k = random.randint(0, 3)
            if k:
                img_p = np.rot90(img_p, k, (1,2))
                lbl_p = np.rot90(lbl_p, k, (1,2))
            img_p = np.ascontiguousarray(img_p)
            lbl_p = np.ascontiguousarray(lbl_p)
            
            if random.random() > 0.5:
                img_p = img_p * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_p = img_p + random.uniform(-0.1, 0.1)
        
        fg_mask = (lbl_p == 1).astype(np.float32)
        ig_mask = (lbl_p == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_p).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }

print("Dataset ready")

In [ ]:
# =============================================================================
# CELL 5: CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    def __init__(self, save_dir, load_dir=None, max_keep=5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch, metrics):
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
        }
        
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None, load_best=False):
        path = self.load_dir / ('best_model.pth' if load_best else 'last_checkpoint.pth')
        if not path.exists():
            print("No checkpoint found, starting fresh")
            return 0
        
        print(f"Loading {path}")
        ckpt = torch.load(path, map_location=cfg.DEVICE, weights_only=False)
        
        # Clean keys from torch.compile
        state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
        model.load_state_dict(state, strict=False)
        
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        if (self.save_dir / 'history.json').exists():
            with open(self.save_dir / 'history.json') as f:
                self.history = json.load(f)
        
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best: {self.best_score:.4f} @ epoch {self.best_epoch}")
        
        return ckpt['epoch'] + 1

print("CheckpointManager ready")

In [ ]:
# =============================================================================
# CELL 6: TRAINING FUNCTIONS
# =============================================================================

import sys

def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss = 0
    total_dice = 0
    n = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}", file=sys.stdout)
    for batch in pbar:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        ignore = batch['ignore_mask'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=cfg.USE_AMP):
            out = model(images)
            losses = criterion(out, labels, ignore, epoch)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        n += 1
        
        pbar.set_postfix(loss=f"{losses['total'].item():.3f}", dice=f"{losses['dice']:.3f}")
    
    return {'train_loss': total_loss/n, 'train_dice': total_dice/n}


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_dice = 0
    n = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            out = model(images)
            if isinstance(out, dict): out = out['output']
            probs = torch.sigmoid(out).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b,0] > 0.5).astype(np.uint8)
            tgt = labels[b,0].astype(np.uint8)
            ign = ignore[b,0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2*inter + 1e-5) / (union + 1e-5)
            n += 1
    
    return {'val_dice': total_dice / max(n, 1)}

print("Training functions ready")

In [ ]:
# =============================================================================
# CELL 7: MAIN TRAINING
# =============================================================================

def main():
    print("="*60)
    print("V8 MULTI-SCALE TRAINING")
    print("="*60)
    
    # Data
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    print("\n[1/4] Loading training data...")
    train_ds = VesuviusDataset(
        train_csv, train_images, train_labels,
        patch_size=cfg.PATCH_SIZE, is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
    )
    
    print("\n[2/4] Loading validation data...")
    val_ds = VesuviusDataset(
        train_csv, train_images, train_labels,
        patch_size=cfg.PATCH_SIZE, is_train=False,
        data_fraction=0.15, patches_per_volume=1,
    )
    
    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=True, prefetch_factor=cfg.PREFETCH_FACTOR,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True,
    )
    
    print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
    print(f"Val: {len(val_ds)} samples")
    
    # Model
    print("\n[3/4] Creating model...")
    model = MultiScaleUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    if hasattr(torch, 'compile'):
        print("Compiling model...")
        model = torch.compile(model, mode='reduce-overhead')
    
    print(f"Parameters: {count_params(model):,}")
    
    # Loss
    criterion = CombinedLoss(
        dice_w=cfg.DICE_WEIGHT, bce_w=cfg.BCE_WEIGHT,
        skel_w=cfg.SKELETON_WEIGHT, cldice_w=cfg.CLDICE_WEIGHT,
        skel_start=cfg.SKELETON_START_EPOCH, skel_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH, cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
    )
    
    # Optimizer + OneCycleLR
    optimizer = torch.optim.SGD(
        model.parameters(), lr=cfg.MAX_LR,
        momentum=0.99, weight_decay=cfg.WEIGHT_DECAY, nesterov=True,
    )
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # Checkpoint
    ckpt_mgr = CheckpointManager(
        save_dir=cfg.CHECKPOINT_DIR,
        load_dir=Path("/kaggle/input/v8-checkpoint/checkpoints") if Path("/kaggle/input/v8-checkpoint").exists() else cfg.CHECKPOINT_DIR,
    )
    
    print("\n[4/4] Checking checkpoint...")
    start_epoch = 0
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, None, scaler)
    
    # OneCycleLR scheduler
    steps_per_epoch = len(train_loader)
    total_steps = cfg.EPOCHS * steps_per_epoch
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=cfg.MAX_LR,
        total_steps=total_steps,
        pct_start=cfg.PCT_START,
        div_factor=cfg.DIV_FACTOR,
        final_div_factor=cfg.FINAL_DIV_FACTOR,
        anneal_strategy='cos',
    )
    
    # Skip scheduler steps if resuming
    if start_epoch > 0:
        for _ in range(start_epoch * steps_per_epoch):
            scheduler.step()
    
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"OneCycleLR: max_lr={cfg.MAX_LR}, {cfg.PCT_START*100:.0f}% warmup")
    print("="*60)
    
    import time
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        
        # Train with scheduler step per batch
        model.train()
        total_loss = 0
        total_dice = 0
        n = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}", file=sys.stdout)
        for batch in pbar:
            images = batch['image'].to(cfg.DEVICE, non_blocking=True)
            labels = batch['label'].to(cfg.DEVICE, non_blocking=True)
            ignore = batch['ignore_mask'].to(cfg.DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with autocast(enabled=cfg.USE_AMP):
                out = model(images)
                losses = criterion(out, labels, ignore, epoch)
            
            scaler.scale(losses['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()  # Step per batch for OneCycleLR
            
            total_loss += losses['total'].item()
            total_dice += losses['dice']
            n += 1
            
            pbar.set_postfix(loss=f"{losses['total'].item():.3f}", lr=f"{scheduler.get_last_lr()[0]:.5f}")
        
        train_metrics = {'train_loss': total_loss/n, 'train_dice': total_dice/n}
        
        # Validate
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        dt = time.time() - t0
        lr = scheduler.get_last_lr()[0]
        
        print(f"Epoch {epoch+1}/{cfg.EPOCHS} | {dt:.1f}s | LR: {lr:.6f} | "
              f"Loss: {train_metrics['train_loss']:.4f}" +
              (f" | Val: {val_metrics['val_dice']:.4f}" if val_metrics['val_dice'] > 0 else ""))
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch, {**train_metrics, **val_metrics})
    
    print("\n" + "="*60)
    print(f"DONE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*60)
    
    return model, ckpt_mgr

print("Ready to train!")
print(f"  Epochs: {cfg.EPOCHS}")
print(f"  Patch: {cfg.PATCH_SIZE}")
print(f"  Batch: {cfg.BATCH_SIZE}")

In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================

cfg.RESUME_TRAINING = True
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 600

model, ckpt_mgr = main()

In [ ]:
# Check status
last = cfg.CHECKPOINT_DIR / 'last_checkpoint.pth'
if last.exists():
    ckpt = torch.load(last, map_location='cpu', weights_only=False)
    print(f"Epoch: {ckpt['epoch']+1}/{cfg.EPOCHS}")
    print(f"Best: {ckpt.get('best_score', 0):.4f} @ epoch {ckpt.get('best_epoch', 0)}")